# Gemma Edit-Discipline LoRA — Colab training (A100-40GB)
Open via GitHub->Colab, set **A100 GPU** runtime, Run All.
Trains the edit-discipline adapter on the 23 byte-faithful rich trajectories (Task 1).
RESUMABLE: checkpoints to Google Drive each epoch; re-run to add epochs. Fits 40GB via NF4 + grad-checkpointing + batch 1.

Spec: specs/gemma-lora-colab-training. Pre-registered: adapter must beat v19's 1/5 on a free Kaggle run before any scored slot. v6/0.12 SACRED.

In [ ]:
# 1. GPU assertion + pinned deps (E4). Stop if <40GB.
import torch, subprocess, sys
assert torch.cuda.is_available(), 'No GPU — set Runtime -> A100 GPU'
total = torch.cuda.mem_get_info()[1]/1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  {total:.0f}GB')
assert total >= 38, f'Need >=40GB A100 for 31B QLoRA; got {total:.0f}GB. Use a smaller base (27b/12b) or stop.'
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
                'transformers>=4.49','peft>=0.14','bitsandbytes>=0.45','accelerate>=1.3','safetensors','datasets'], check=True)
print('deps installed')

In [ ]:
# 2. Mount Drive (resumable checkpoints survive disconnect) + clone repo (code + trajectory data).
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/gemma_lora/edit_discipline_r32'
os.makedirs(OUT, exist_ok=True)
print('checkpoint dir:', OUT)
!rm -rf /content/Gemma_Competition
!git clone --depth 1 https://github.com/GeorgeRCAdamJohnson/Gemma_Competition.git /content/Gemma_Competition
%cd /content/Gemma_Competition
!wc -l train/data/edit_discipline_rich.jsonl

In [ ]:
# 3. HF auth for the gated bf16 base. Set HF_TOKEN in Colab Secrets (key icon, name=HF_TOKEN) OR paste below.
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as e:
    print('Set HF_TOKEN in Colab Secrets (recommended), or uncomment next line:')
    # os.environ['HF_TOKEN'] = 'hf_...'
from huggingface_hub import login
login(token=os.environ.get('HF_TOKEN'))
# BASE: try 31b bf16; fall back to 27b/12b if gated/unreachable or OOM. Confirm from probe E2.
BASE = os.environ.get('LORA_BASE', 'google/gemma-4-27b-it')  # 27b safer on 40GB than 31b; override via env if 31b reachable
print('training base:', BASE)

In [ ]:
# 4. TRAIN (resumable). train_lora.py: QLoRA, loss-masked to assistant turns, checkpoints each epoch to OUT.
# r=32 (slice to 16/8 later via make_variant.py). batch is implicit 1 in the script; maxlen 1024 fits the terse trajectories.
# Re-run this cell to RESUME (reads progress.json) and raise :3 -> :5 to add epochs.
!python train/train_lora.py \
  --out {OUT} \
  --base {BASE} \
  --r 32 --alpha 64 --maxlen 1024 --lr 2e-4 \
  --stage s1=data/edit_discipline_rich.jsonl:3

In [ ]:
# 5. Package the adapter: zip from Drive + print sha256/size for the manifest. Download via the Files pane,
#    or it's already on your Drive at OUT. This adapter dir drops into submit_v19_lora/adapters/.
import os, hashlib, glob
files = [f for f in glob.glob(OUT+'/*') if os.path.isfile(f)]
print('adapter files:')
for f in files:
    h = hashlib.sha256(open(f,'rb').read()).hexdigest()[:16]
    print(f'  {os.path.basename(f):30s} {os.path.getsize(f)/1e6:7.1f}MB  sha256={h}')
!cd {OUT} && zip -qr /content/edit_discipline_r32.zip . && ls -la /content/edit_discipline_r32.zip
print('\nAdapter is on Drive at', OUT, 'and zipped at /content/edit_discipline_r32.zip (download from Files pane).')